In [ ]:
# Notebook 04: Xay dung, danh gia va luu recommendation models

In [1]:
import ast
import json
import joblib
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import sparse
from scipy.sparse import csr_matrix
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity
from collections import defaultdict

FEATURES_DIR = Path("../artifacts/features")

with open(FEATURES_DIR / "feature_manifest.json", encoding="utf-8") as f:
    manifest = json.load(f)

tfidf_vectorizer = joblib.load(FEATURES_DIR / "tfidf_vectorizer.joblib")
recipe_tfidf_matrix = sparse.load_npz(FEATURES_DIR / "recipe_tfidf_matrix.npz")

recipe_index_map = pd.read_csv(FEATURES_DIR / "recipe_index_map.csv")
recipe_similarity_topk = pd.read_csv(FEATURES_DIR / "recipe_similarity_topk.csv")
recipe_features = pd.read_csv(FEATURES_DIR / "recipe_features.csv")
popularity_features = pd.read_csv(FEATURES_DIR / "popularity_features.csv")

user_features = pd.read_csv(FEATURES_DIR / "user_features.csv")
user_features["favorite_tags"] = user_features["favorite_tags"].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) and x.startswith("[") else []
)

train_interactions = pd.read_csv(FEATURES_DIR / "train_interactions.csv")
val_interactions = pd.read_csv(FEATURES_DIR / "val_interactions.csv")
train_interactions["date"] = pd.to_datetime(train_interactions["date"], errors="coerce")
val_interactions["date"] = pd.to_datetime(val_interactions["date"], errors="coerce")

df_recipes = pd.read_csv("../data/processed/recipes_clean.csv", usecols=["id", "name"])
recipe_name_map = dict(zip(df_recipes["id"], df_recipes["name"]))

id2idx = dict(zip(recipe_index_map["recipe_id"], recipe_index_map["matrix_index"]))
idx2id = dict(zip(recipe_index_map["matrix_index"], recipe_index_map["recipe_id"]))

train_user_ids = set(train_interactions["user_id"].unique())
train_recipe_ids = set(train_interactions["recipe_id"].unique())

print("manifest:", manifest.get("version"), manifest.get("build_timestamp"))
print("matrix:", recipe_tfidf_matrix.shape, "| train:", len(train_interactions), "| val:", len(val_interactions))

manifest: 1.0 2026-05-01T08:32:33.084922
matrix: (76608, 20000) | train: 559500 | val: 139876


In [ ]:
# Goi y theo do pho bien

popularity_ranked = popularity_features.sort_values("popularity_score", ascending=False).reset_index(drop=True)

def recommend_popular(n=10, exclude_ids=None):
    df = popularity_ranked
    if exclude_ids:
        df = df[~df["recipe_id"].isin(exclude_ids)]
    top = df.head(n).copy()
    top["rank"] = range(1, len(top) + 1)
    top["name"] = top["recipe_id"].map(recipe_name_map)
    return top[["rank", "recipe_id", "name", "popularity_score"]].reset_index(drop=True)

display(recommend_popular(10))

,rank,recipe_id,name,popularity_score
0,1,77497,2 handed kitchen sink tomato sandwich,4.969171
1,2,34753,basic sourdough bread,4.967149
2,3,154351,kittencal s balsamic vinaigrette,4.965450
3,4,78579,pan release professional pan coating better th...,4.965094
4,5,186029,the best creole cajun seasoning mix,4.964216
5,6,58516,hot pepper jelly,4.961464
6,7,244193,absolutely the best new york cheesecake gluten...,4.959104
7,8,55309,caprese salad tomatoes italian marinated tomatoes,4.958252
8,9,31639,7 layer bean dip,4.957364
9,10,25094,my chicken parmigiana,4.957087


In [ ]:
# Content-based: cong similarity tu neighbor cua cac recipe user da rating >= 4

neighbors_lookup = recipe_similarity_topk.groupby("recipe_id")

def recommend_content_similar(recipe_id, n=10):
    if recipe_id not in neighbors_lookup.groups:
        return pd.DataFrame()

    top = (
        neighbors_lookup.get_group(recipe_id)
        .sort_values("similarity", ascending=False)
        .head(n)
        .copy()
    )
    top["name"] = top["neighbor_recipe_id"].map(recipe_name_map)
    top["rank"] = range(1, len(top) + 1)
    return top.rename(columns={"neighbor_recipe_id": "recipe_id"}).loc[:, ["rank", "recipe_id", "name", "similarity"]]


def recommend_for_user_content(user_id, n=10):
    hist = train_interactions[train_interactions["user_id"] == user_id]
    seen = set(hist["recipe_id"])
    liked = hist.loc[hist["rating"] >= 4, "recipe_id"].tolist()

    if not liked:
        return recommend_popular(n, exclude_ids=seen if seen else None)

    scores = defaultdict(list)
    for rid in liked:
        if rid not in neighbors_lookup.groups:
            continue
        for idx, row in neighbors_lookup.get_group(rid).iterrows():
            scores[row["neighbor_recipe_id"]].append(row["similarity"])

    scored = {
        r: float(np.mean(v))
        for r, v in scores.items()
        if r not in seen
    }
    ranked = sorted(scored.items(), key=lambda x: x[1], reverse=True)[:n]

    out = pd.DataFrame(ranked, columns=["recipe_id", "content_score"])
    out["name"] = out["recipe_id"].map(recipe_name_map)
    out["rank"] = range(1, len(out) + 1)
    return out[["rank", "recipe_id", "name", "content_score"]]

In [ ]:
# CF: SVD tren ma tran user-item 

user_ids = train_interactions["user_id"].unique()
recipe_ids = train_interactions["recipe_id"].unique()

user_to_index = {u: i for i, u in enumerate(user_ids)}
item_to_index = {r: i for i, r in enumerate(recipe_ids)}

rows = train_interactions["user_id"].map(user_to_index).to_numpy()
cols = train_interactions["recipe_id"].map(item_to_index).to_numpy()
ratings = train_interactions["rating"].to_numpy()

user_item_matrix = csr_matrix((ratings, (rows, cols)), shape=(len(user_ids), len(recipe_ids)))

user_means = train_interactions.groupby("user_id")["rating"].mean()
centered_vals = ratings - train_interactions["user_id"].map(user_means).to_numpy()
centered_matrix = csr_matrix((centered_vals, (rows, cols)), shape=user_item_matrix.shape)

svd_model = TruncatedSVD(n_components=50, random_state=42)
user_factors = svd_model.fit_transform(centered_matrix)
item_factors = svd_model.components_

item_ids_array = np.array(recipe_ids)
user_seen_lookup = train_interactions.groupby("user_id")["recipe_id"].apply(set).to_dict()

def recommend_cf(user_id, n=10):
    if user_id not in user_to_index:
        return recommend_popular(n)

    u_idx = user_to_index[user_id]
    scores = user_factors[u_idx] @ item_factors + user_means[user_id]

    seen = user_seen_lookup.get(user_id, set())
    if seen:
        candidate_mask = ~np.isin(item_ids_array, list(seen))
        candidate_indices = np.where(candidate_mask)[0]
    else:
        candidate_indices = np.arange(len(item_ids_array))

    if len(candidate_indices) == 0:
        return pd.DataFrame(columns=["rank", "recipe_id", "name", "predicted_rating"])

    candidate_scores = scores[candidate_indices]
    top_k = min(n, len(candidate_indices))

    if top_k == len(candidate_indices):
        top_local = np.argsort(-candidate_scores)
    else:
        top_local = np.argpartition(-candidate_scores, top_k - 1)[:top_k]
        top_local = top_local[np.argsort(-candidate_scores[top_local])]

    top_indices = candidate_indices[top_local]
    out = pd.DataFrame({
        "recipe_id": item_ids_array[top_indices],
        "predicted_rating": candidate_scores[top_local],
    })
    out["name"] = out["recipe_id"].map(recipe_name_map)
    out["rank"] = range(1, len(out) + 1)
    return out[["rank", "recipe_id", "name", "predicted_rating"]]

In [9]:
# Loc ket qua theo calories/phut/so nguyen lieu/tags

df_recipes_full = pd.read_csv("../data/processed/recipes_clean.csv", usecols=["id", "tags_clean"])

def _parse_tags(x):
    if isinstance(x, list):
        return x
    if isinstance(x, str):
        try:
            return ast.literal_eval(x)
        except Exception:
            return []
    return []

df_recipes_full["tags_clean"] = df_recipes_full["tags_clean"].apply(_parse_tags)
recipe_tags_idx = df_recipes_full.set_index("id")["tags_clean"]

def apply_constraints(candidate_df, max_calories=None, max_minutes=None,
                      tags_include=None, max_ingredients=None):
    feats = recipe_features[["recipe_id", "calories", "minutes", "n_ingredients_calc"]]
    df = candidate_df.merge(feats, on="recipe_id", how="left", suffixes=("", "_feat"))

    for col in ["calories", "minutes", "n_ingredients_calc"]:
        scol = col + "_feat"
        if scol in df.columns:
            df[col] = df[col].combine_first(df[scol])
            df = df.drop(columns=[scol], errors="ignore")

    if max_calories is not None:
        df = df[df["calories"] <= max_calories]
    if max_minutes is not None:
        df = df[df["minutes"] <= max_minutes]
    if max_ingredients is not None:
        df = df[df["n_ingredients_calc"] <= max_ingredients]

    if tags_include:
        tags_set = set(tags_include)
        df = df[df["recipe_id"].map(recipe_tags_idx).apply(lambda t: tags_set.issubset(set(t) if isinstance(t, list) else set()))]

    df = df.drop(columns=["calories", "minutes", "n_ingredients_calc"], errors="ignore").reset_index(drop=True)
    df["rank"] = range(1, len(df) + 1)
    return df

In [10]:
# Hybrid: noi content + CF, alpha theo muc hoat dong, fallback popularity

def min_max_normalize(series):
    mn, mx = series.min(), series.max()
    if mx == mn:
        return pd.Series(0.5, index=series.index)
    return (series - mn) / (mx - mn)

def hybrid_pop_fallback(seen_ids, n):
    out = recommend_popular(n=n * 3, exclude_ids=seen_ids)
    out["source"] = "popularity"
    return out.head(n).reset_index(drop=True)

def recommend_hybrid(user_id, n=10, constraints=None):
    seen_ids = set(train_interactions.loc[train_interactions["user_id"] == user_id, "recipe_id"])

    if user_id not in train_user_ids:
        out = hybrid_pop_fallback(seen_ids, n)
        return apply_constraints(out, **constraints) if constraints else out

    user_row = user_features[user_features["user_id"] == user_id]
    if user_row.empty:
        out = hybrid_pop_fallback(seen_ids, n)
        return apply_constraints(out, **constraints) if constraints else out

    rating_count = int(user_row["rating_count"].iloc[0])
    alpha = 0.7 if rating_count < 5 else 0.5

    content_results = recommend_for_user_content(user_id, n=n * 5)
    cf_results = recommend_cf(user_id, n=n * 5)

    has_content = "content_score" in content_results.columns
    has_cf = "predicted_rating" in cf_results.columns

    if content_results.empty and cf_results.empty:
        out = hybrid_pop_fallback(seen_ids, n)
        return apply_constraints(out, **constraints) if constraints else out

    if not has_content and not has_cf:
        out = hybrid_pop_fallback(seen_ids, n)
        return apply_constraints(out, **constraints) if constraints else out

    if not has_content:
        out = cf_results.copy()
        out["source"] = "cf"
        out = out.head(n).reset_index(drop=True)
        return apply_constraints(out, **constraints) if constraints else out

    if not has_cf:
        out = content_results.copy()
        out["source"] = "content"
        out = out.head(n).reset_index(drop=True)
        return apply_constraints(out, **constraints) if constraints else out

    content_df = content_results[["recipe_id", "content_score"]]
    cf_df = cf_results[["recipe_id", "predicted_rating"]].rename(columns={"predicted_rating": "cf_score"})

    merged = content_df.merge(cf_df, on="recipe_id", how="outer")
    merged["content_score"] = merged["content_score"].fillna(merged["content_score"].median())
    merged["cf_score"] = merged["cf_score"].fillna(merged["cf_score"].median())

    merged["content_norm"] = min_max_normalize(merged["content_score"])
    merged["cf_norm"] = min_max_normalize(merged["cf_score"])
    merged["hybrid_score"] = alpha * merged["content_norm"] + (1 - alpha) * merged["cf_norm"]

    merged = merged.sort_values("hybrid_score", ascending=False).reset_index(drop=True)
    merged["rank"] = range(1, len(merged) + 1)
    merged["name"] = merged["recipe_id"].map(recipe_name_map)
    merged["source"] = "hybrid"

    result = merged[["rank", "recipe_id", "name", "hybrid_score", "content_norm", "cf_norm", "source"]]

    if constraints:
        result = apply_constraints(result, **constraints)

    return result.head(n).reset_index(drop=True)

In [12]:
# Xem nhanh goi y cho 1 user active / warm / cold

sample_active = user_features[user_features["activity_level"] == "active"]["user_id"].iloc[0]
sample_warm = user_features[user_features["activity_level"] == "warm"]["user_id"].iloc[0]
cold_users = set(val_interactions["user_id"]) - train_user_ids
sample_cold = next(iter(cold_users))  # neu rong se loi: can if cold_users

sample_users = [
    (sample_active, "active"),
    (sample_warm, "warm"),
    (sample_cold, "cold"),
]

for user_id, nhom in sample_users:
    print(f"\nuser={user_id} | {nhom}")

    hist = train_interactions[train_interactions["user_id"] == user_id]
    liked = hist[hist["rating"] >= 4]
    print(f"tuong tac train: {len(hist)} | thich (>=4): {len(liked)}")
    if not liked.empty:
        print("vi du mon thich:", liked["recipe_id"].head(5).map(recipe_name_map).tolist())

    seen = set(hist["recipe_id"]) if not hist.empty else None

    print("\npopularity")
    display(recommend_popular(5, exclude_ids=seen))

    print("\ncontent")
    display(recommend_for_user_content(user_id, n=5))

    print("\ncf")
    cf = recommend_cf(user_id, n=5)
    display(cf) if not cf.empty else print("rong")

    print("\nhybrid")
    display(recommend_hybrid(user_id, n=5))


user=1533 | active
tuong tac train: 73 | thich (>=4): 71
vi du mon thich: ['zucchini and rice casserole', 'orange yogurt cream', 'parmesan fish in the oven', 'fennel mashed potatoes', 'c hudson s ham potato casserole']

popularity


,rank,recipe_id,name,popularity_score
0,1,77497,2 handed kitchen sink tomato sandwich,4.969171
1,2,34753,basic sourdough bread,4.967149
2,3,154351,kittencal s balsamic vinaigrette,4.965450
3,4,78579,pan release professional pan coating better th...,4.965094
4,5,186029,the best creole cajun seasoning mix,4.964216



content


,rank,recipe_id,name,content_score
0,1,46277.0,no salt seasoning,0.702102
1,2,68017.0,italian seasoning,0.597112
2,3,4029.0,herb blend for boursin cream cheese,0.583898
3,4,48555.0,memphis rub,0.569698
4,5,83902.0,montreal steak seasoning,0.551078



cf


...

,rank,recipe_id,name,popularity_score
0,1,77497,2 handed kitchen sink tomato sandwich,4.969171
1,2,34753,basic sourdough bread,4.967149
2,3,154351,kittencal s balsamic vinaigrette,4.965450
3,4,78579,pan release professional pan coating better th...,4.965094
4,5,186029,the best creole cajun seasoning mix,4.964216



cf


,rank,recipe_id,name,popularity_score
0,1,77497,2 handed kitchen sink tomato sandwich,4.969171
1,2,34753,basic sourdough bread,4.967149
2,3,154351,kittencal s balsamic vinaigrette,4.965450
3,4,78579,pan release professional pan coating better th...,4.965094
4,5,186029,the best creole cajun seasoning mix,4.964216



hybrid


,rank,recipe_id,name,popularity_score,source
0,1,77497,2 handed kitchen sink tomato sandwich,4.969171,popularity
1,2,34753,basic sourdough bread,4.967149,popularity
2,3,154351,kittencal s balsamic vinaigrette,4.965450,popularity
3,4,78579,pan release professional pan coating better th...,4.965094,popularity
4,5,186029,the best creole cajun seasoning mix,4.964216,popularity


In [13]:
import time

K = 10

val_positive = val_interactions[val_interactions["rating"] >= 4].copy()
train_seen_by_user = train_interactions.groupby("user_id")["recipe_id"].apply(set).to_dict()

def build_unseen_relevant(group):
    seen = train_seen_by_user.get(group.name, set())
    return set(group[~group.isin(seen)])

val_relevant = (
    val_positive.groupby("user_id")["recipe_id"]
    .apply(build_unseen_relevant)
    .to_dict()
)
val_relevant = {u: s for u, s in val_relevant.items() if s}
val_user_ids = list(val_relevant.keys())

print("user val co relevant unseen:", len(val_user_ids))
print("avg relevant/user:", round(np.mean([len(v) for v in val_relevant.values()]), 2))

def precision_at_k(rec, rel, k):
    s = set(rec[:k])
    return len(s & rel) / k if k else 0.0

def recall_at_k(rec, rel, k):
    s = set(rec[:k])
    return len(s & rel) / len(rel) if rel else 0.0

def ndcg_at_k(rec, rel, k):
    dcg = sum((1 / np.log2(i + 2)) for i, x in enumerate(rec[:k]) if x in rel)
    nhits = min(len(rel), k)
    idcg = sum(1 / np.log2(i + 2) for i in range(nhits))
    return dcg / idcg if idcg else 0.0

def evaluate_model(fn, name, users, k=K, max_error_rate=0.01):
    ps, rs, ds = [], [], []
    all_rec = set()
    skipped, errors = 0, 0
    t0 = time.time()

    for i, uid in enumerate(users):
        rel = val_relevant.get(uid, set())
        if not rel:
            skipped += 1
            continue

        try:
            out = fn(uid, n=k)
            if not isinstance(out, pd.DataFrame) or "recipe_id" not in out.columns:
                raise ValueError("can thi cot recipe_id")
            rec = out["recipe_id"].tolist()
        except Exception as e:
            errors += 1
            if errors <= 5:
                print(f"{name}: loi user {uid}: {e}")
            continue

        ps.append(precision_at_k(rec, rel, k))
        rs.append(recall_at_k(rec, rel, k))
        ds.append(ndcg_at_k(rec, rel, k))
        all_rec.update(rec)

        if (i + 1) % 2000 == 0:
            print(f"{name}: {i+1}/{len(users)} | {time.time()-t0:.0f}s | loi={errors}")

    elapsed = time.time() - t0
    n_ok = len(ps)
    tried = n_ok + errors
    err_rate = (errors / tried) if tried else 0.0

    if err_rate > max_error_rate:
        raise RuntimeError(f"{name}: ti le loi qua cao ({err_rate:.2%})")

    cov = len(all_rec) / len(recipe_name_map) if recipe_name_map else 0.0

    m = {
        "model": name,
        "P@K": float(np.mean(ps)) if ps else 0.0,
        "R@K": float(np.mean(rs)) if rs else 0.0,
        "NDCG@K": float(np.mean(ds)) if ds else 0.0,
        "Coverage": cov,
        "users_evaluated": n_ok,
        "users_skipped": skipped,
        "errors": errors,
        "error_rate": round(err_rate, 6),
        "time_seconds": round(elapsed, 1),
    }
    print(f"{name}: xong {elapsed:.1f}s | P@{k}={m['P@K']:.4f} | loi={errors}")
    return m

def recommend_popularity_for_user(user_id, n=10):
    return recommend_popular(n=n, exclude_ids=train_seen_by_user.get(user_id, set()))

print(f"\nDanh gia val, K={K}\n")
results = []
results.append(evaluate_model(recommend_popularity_for_user, "Popularity", val_user_ids))
results.append(evaluate_model(recommend_for_user_content, "Content", val_user_ids))
results.append(evaluate_model(recommend_cf, "CF (SVD)", val_user_ids))
results.append(evaluate_model(recommend_hybrid, "Hybrid", val_user_ids))

comparison = pd.DataFrame(results)[
    ["model", "P@K", "R@K", "NDCG@K", "Coverage", "users_evaluated", "time_seconds"]
]
display(comparison)

cold_users = [u for u in val_user_ids if u not in train_user_ids]
print("\ncold-start trong val:", len(cold_users))
if cold_users:
    cold_rows = []
    for name, fn in [
        ("Popularity", recommend_popularity_for_user),
        ("Content", recommend_for_user_content),
        ("Hybrid", recommend_hybrid),
    ]:
        cold_rows.append(evaluate_model(fn, name, cold_users))
    display(pd.DataFrame(cold_rows)[["model", "P@K", "R@K", "NDCG@K", "Coverage"]])

user val co relevant unseen: 14129
avg relevant/user: 9.38

Danh gia val, K=10

Popularity: 2000/14129 | 67s | loi=0
Popularity: 4000/14129 | 144s | loi=0
Popularity: 6000/14129 | 216s | loi=0
Popularity: 8000/14129 | 278s | loi=0
Popularity: 10000/14129 | 341s | loi=0
Popularity: 12000/14129 | 403s | loi=0
Popularity: 14000/14129 | 461s | loi=0
Popularity: xong 465.0s | P@10=0.0009 | loi=0
Content: 2000/14129 | 329s | loi=0
Content: 4000/14129 | 548s | loi=0
Content: 6000/14129 | 769s | loi=0
Content: 8000/14129 | 921s | loi=0
Content: 10000/14129 | 1046s | loi=0
Content: 12000/14129 | 1146s | loi=0
Content: 14000/14129 | 1219s | loi=0
Content: xong 1227.3s | P@10=0.0006 | loi=0
CF (SVD): 2000/14129 | 80s | loi=0
CF (SVD): 4000/14129 | 157s | loi=0
CF (SVD): 6000/14129 | 335s | loi=0
CF (SVD): 8000/14129 | 448s | loi=0
CF (SVD): 10000/14129 | 544s | loi=0
CF (SVD): 12000/14129 | 633s | loi=0
CF (SVD): 14000/14129 | 711s | loi=0
CF (SVD): xong 715.8s | P@10=0.0023 | loi=0
Hybrid: 2000/

,model,P@K,R@K,NDCG@K,Coverage,users_evaluated,time_seconds
0,Popularity,0.000878,0.001366,0.001254,0.000196,14129,465.0
1,Content,0.000602,0.000808,0.000842,0.329874,14129,1227.3
2,CF (SVD),0.002350,0.004176,0.004037,0.017152,14129,715.8
3,Hybrid,0.001600,0.002880,0.002694,0.306926,14129,2938.8



cold-start trong val: 3787
Popularity: 2000/3787 | 97s | loi=0
Popularity: xong 212.9s | P@10=0.0007 | loi=0
Content: 2000/3787 | 93s | loi=0
Content: xong 165.5s | P@10=0.0007 | loi=0
Hybrid: 2000/3787 | 85s | loi=0
Hybrid: xong 165.7s | P@10=0.0007 | loi=0


,model,P@K,R@K,NDCG@K,Coverage
0,Popularity,0.000713,0.001096,0.000879,0.000131
1,Content,0.000713,0.001096,0.000879,0.000131
2,Hybrid,0.000713,0.001096,0.000879,0.000131


In [ ]:
import json
import joblib
from pathlib import Path
from datetime import datetime

MODELS_DIR = Path("../artifacts/models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

#SVD artifact
svd_artifact = {
    "svd_model": svd_model,
    "user_factors": user_factors,
    "item_factors": item_factors,
    "user_means": user_means.to_dict(),
}
joblib.dump(svd_artifact, MODELS_DIR / "svd_model.joblib")

#User index map
user_index_df = pd.DataFrame(
    [{"user_id": uid, "matrix_index": idx} for uid, idx in user_to_index.items()]
).sort_values("matrix_index")
user_index_df.to_csv(MODELS_DIR / "user_index_map.csv", index=False)

#Item index map 
item_index_df = pd.DataFrame(
    [{"recipe_id": rid, "matrix_index": idx} for rid, idx in item_to_index.items()]
).sort_values("matrix_index")
item_index_df.to_csv(MODELS_DIR / "item_index_map.csv", index=False)

#Hybrid config
hybrid_config = {
    "version": "1.0",
    "build_timestamp": datetime.now().isoformat(),
    "svd_n_components": 50,
    "tiers": {
        "cold_start": {"condition": "user not in train", "strategy": "popularity"},
        "warm": {"condition": "rating_count < 5", "alpha": 0.7, "strategy": "content-heavy hybrid"},
        "active": {"condition": "rating_count >= 5", "alpha": 0.5, "strategy": "balanced hybrid"},
    },
    "constraint_defaults": {
        "max_calories": None,
        "max_minutes": None,
        "tags_include": None,
        "max_ingredients": None,
    },
    "data_stats": {
        "train_users": len(train_user_ids),
        "train_interactions": len(train_interactions),
        "val_interactions": len(val_interactions),
        "total_recipes": len(recipe_name_map),
    },
}
with open(MODELS_DIR / "hybrid_config.json", "w", encoding="utf-8") as f:
    json.dump(hybrid_config, f, indent=2, ensure_ascii=False)

#Evaluation results
evaluation_output = {
    "K": K,
    "timestamp": datetime.now().isoformat(),
    "all_users": results,
}
if "cold_results" in globals() and cold_results:
    evaluation_output["cold_start_users"] = cold_results

with open(MODELS_DIR / "evaluation_results.json", "w", encoding="utf-8") as f:
    json.dump(evaluation_output, f, indent=2, ensure_ascii=False)

print("Da luu vao:", MODELS_DIR.resolve())
print("Files:", ["svd_model.joblib", "user_index_map.csv", "item_index_map.csv", "hybrid_config.json", "evaluation_results.json"])

Da luu vao: D:\GITHUB\Food-RecommendationSystem\artifacts\models
Files: ['svd_model.joblib', 'user_index_map.csv', 'item_index_map.csv', 'hybrid_config.json', 'evaluation_results.json']
